# 02 — CAPM & the Beta-Target Portfolio

Notebook 01 showed the practical failure mode of mean-variance: fed a noisy
252-day sample-mean $\mu$, `MaxSharpe` becomes an error-maximizer, jumping to
near-single-asset corners with no better realized Sharpe than `EqualWeight` to
show for it. CAPM offers a way to get $\mu$ *without* estimating sample means
at all — $\mu - r_f = \beta(\mu_{mkt} - r_f)$, a single market price of risk
times a stable-looking loading. This notebook checks that promise on the same
13-asset universe: it turns out CAPM $\mu$ has the same problem as sample-mean
$\mu$, for a related reason (§3) — but $\beta$ itself, used directly as a
*constraint* rather than routed through $\mu$, gives a workable allocation
rule (§5).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt

import maplab as ml
import _nb02_helpers as h
from maplab import Panel, apply_style

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")

## CAPM on the 13 assets

$\text{MKT} = r_{SPY} - r_f$, daily **log** excess returns (BIL's simple
return converted to log space via $\log(1+r_f)$, so it is directly
comparable/subtractable from the log asset returns). Full-window $\beta_i$,
realized annualized excess return, and $R^2$ per asset — a plain OLS slope
($\beta = \mathrm{cov}(r_i-r_f,\ \text{MKT})/\mathrm{var}(\text{MKT})$, which
is exactly the OLS slope of a regression with an intercept, regardless of
what the intercept happens to be).

In [ ]:
rf_log = np.log1p(rf_daily)
excess = log_returns.sub(rf_log, axis=0).dropna(how="any")
MKT = excess["SPY"]

capm_rows = []
for tkr in ml.UNIVERSE:
    fit = ml.inference.ols(excess[tkr], MKT.to_frame("MKT"))
    ann_excess = float(excess[tkr].mean() * ml.TRADING_DAYS)
    capm_rows.append({"ticker": tkr, "beta": fit["coef"]["MKT"],
                       "ann_excess_return": ann_excess, "r2": fit["r2"]})

capm_table = pd.DataFrame(capm_rows).set_index("ticker")
print(f"SPY beta (should be exactly 1.0): {capm_table.loc['SPY', 'beta']:.10f}")
capm_table.round(4)

**Security market line — one historical path, no confidence intervals.**
The dashed line is the *empirical* cross-sectional fit (OLS of realized
excess return on $\beta$ across the 13 assets, this one sample); the dotted
line is the *theoretical* CAPM line through the origin and SPY's own point
(since $\beta_{SPY}=1$ by construction). Neither has a standard error attached
— this is a single 17-year draw, not a distribution.

In [ ]:
fig, slope, intercept, spy_excess_ann = h.sml_figure(capm_table)
plt.show()

print(f"Empirical SML: slope={slope:.4f}, intercept={intercept:.4f}")
print(f"CAPM (theoretical) slope = SPY's realized annualized excess return = {spy_excess_ann:.4f}")
print(f"The empirical slope is {slope / spy_excess_ann:.1%} of the CAPM slope, with a positive "
      f"intercept ({intercept:.2%}) instead of the CAPM-predicted 0 -- a flat, low-beta-anomaly-shaped SML.")

**Rolling 252d $\beta$** for a handful of assets (SPY is fixed at 1 by
construction, plotted as a reference line, not rolled): $\beta$ moves
substantially over the sample, including sign flips for the flight-to-quality
assets — a single full-window number hides a lot of regime-dependence.

In [ ]:
fig = h.rolling_beta_figure(excess, MKT)
plt.show()

## Why CAPM $\mu$ cannot be the allocation rule

If $\mu - r_f = B\lambda$ where $B$ is the (13 x k) matrix of factor loadings
and the factors are themselves portfolios of the 13 universe assets ($F = XW$
for some 13 x k loading matrix $W$, e.g. $W=e_{SPY}$ for a single-asset MKT
factor), then $B = \Sigma_x W (W'\Sigma_x W)^{-1}$ exactly (the OLS slope
identity), so:

$$\Sigma_x^{-1}(\mu-r_f) = \Sigma_x^{-1} B\lambda = W(W'\Sigma_x W)^{-1}\lambda$$

For single-factor CAPM, $W$ is literally the one-hot vector $e_{SPY}$, so the
unconstrained tangency portfolio **degenerates to exactly 100% SPY** — not an
estimation artifact, an algebraic certainty. Adding TERM and CREDIT (both
still portfolios of universe assets: TERM=TLT, CREDIT=HYG-IEF) only expands
the tangency's support to the assets *inside* the factors themselves. CAPM
(or any factor model whose factors are spanned by the universe) can't route
information to assets the factors don't touch — it isn't the sample-mean
error-maximization from notebook 01, but the failure mode is just as total.

In [ ]:
ASOF_DATES = ["2014-12-31", "2017-12-31", "2022-12-31"]
TICKERS = ml.UNIVERSE
N = len(TICKERS)

for res in h.capm_collapse(log_returns, excess, ASOF_DATES):
    asof, designs, rel1, rel2 = res["asof"], res["designs"], res["rel1"], res["rel2"]
    print(f"=== {asof.date()} ===")
    for name, w in designs.items():
        w_s = pd.Series(w, index=TICKERS)
        nz = w_s[w_s.abs() > 1e-4]
        parts = "  ".join(f"{t}={v * 100:+.2f}%" for t, v in nz.items())
        print(f"  {name}: {parts}")

    print(f"  algebra check (relative error ||Σx⁻¹Bλ - W(W'ΣxW)⁻¹λ|| / ||Σx⁻¹Bλ||):  "
          f"A1={rel1:.2e}   A2={rel2:.2e}")
    print()

**Conclusion:** CAPM $\mu$ is the natural *prior* to combine with views
(notebook 03, Black-Litterman) — not an allocation rule on its own.

## Factor diagnostics: MKT / SIZE / VALUE

SIZE = IWM $-$ SPY, VALUE = IWD $-$ IWF (`ml.load_factor_returns()`, panel-only
factor ETFs never allocatable — see contract.FACTOR_TICKERS). Full-window
time-series regression of each asset's excess return on all three factors:
loadings, $t$-stats, $R^2$.

In [ ]:
factor_log_returns = ml.load_factor_returns()
SIZE = (factor_log_returns["IWM"] - log_returns["SPY"]).dropna()
VALUE = (factor_log_returns["IWD"] - factor_log_returns["IWF"]).dropna()

factors3 = pd.DataFrame({"MKT": MKT, "SIZE": SIZE, "VALUE": VALUE}).dropna()
excess3 = excess.loc[factors3.index, TICKERS]

factor_rows = []
for tkr in TICKERS:
    fit = ml.inference.ols(excess3[tkr], factors3)
    row = {"ticker": tkr, "r2": fit["r2"]}
    for f in ["MKT", "SIZE", "VALUE"]:
        row[f"beta_{f}"] = fit["coef"][f]
        row[f"t_{f}"] = fit["tstat"][f]
    factor_rows.append(row)

factor_table = pd.DataFrame(factor_rows).set_index("ticker")
factor_table = factor_table[["beta_MKT", "t_MKT", "beta_SIZE", "t_SIZE", "beta_VALUE", "t_VALUE", "r2"]]
# SPY regressed on MKT=SPY-BIL is degenerate (R^2=1, SPY on itself) -- its
# t-stat is numerically meaningless, not a real factor loading.
factor_table.loc["SPY", "t_MKT"] = np.nan
factor_table.round(3)

In [ ]:
sig = factor_table.copy()
insig_size = sig.index[sig["t_SIZE"].abs() < 2].tolist()
insig_value = sig.index[sig["t_VALUE"].abs() < 2].tolist()
print(f"|t| < 2 on SIZE ({len(insig_size)}/{N}): {insig_size}")
print(f"|t| < 2 on VALUE ({len(insig_value)}/{N}): {insig_value}")

**SIZE** is statistically insignificant on 9/13 assets (|t|<2) — plausible on
priors, a domestic small-cap tilt has no reason to price a Treasury bond or
the dollar.

**VALUE is a different story: it is significant on 11/13 assets**, including
*negative* loadings on rates (IEF, TLT) and *positive* loadings on DBC and
HYG. IWD$-$IWF isn't cleanly "value" outside equities — it carries a **duration**
component (growth stocks trade like long-duration assets, so value-minus-growth
ends up negatively correlated with duration-heavy long bonds, and positively
with credit/commodities that move opposite duration in a rate selloff). These
are plain OLS $t$-stats on ~4,300 daily observations with **no HAC
correction**, so they overstate precision — read them as indicative of an
economic story, not as exact significance tests.

**Cautionary cell — design D3**: long-only minimum-variance subject to
$B'w = [0.5, 0.1, 0.1]$ on MKT/SIZE/VALUE (252d $B$), at the same three dates.
The lesson isn't that these loadings are zero — VALUE in particular is not —
it's that **252-day-window loadings are unstable and, outside equities,
often reflect a proxy relationship (duration, not literal "value") with weak
or shifting economic meaning**. Pinning a portfolio to them with *exact*
equality constraints forces the optimizer to corner-hop between whichever
asset happens to satisfy that quarter's noisy loading — the same
corner-hopping instability as `MaxSharpe`'s noisy $\mu$, from a different
noisy input.

In [ ]:
for asof, w, ok, msg in h.design_d3(log_returns, excess, factors3, ASOF_DATES):
    print(f"=== {asof.date()} ===  {'OK' if ok else 'INFEASIBLE: ' + msg}")
    if ok:
        w_s = pd.Series(w, index=TICKERS)
        nz = w_s[w_s.abs() > 1e-4]
        print("  " + "  ".join(f"{t}={v * 100:+.2f}%" for t, v in nz.items()))

## The allocation: BetaTargetMinVar

CAPM's logic, used correctly: $\beta$ is the only source of risk CAPM says is
*priced*, so fix the $\beta$ you want and minimize every other (unpriced)
source of variance. There is no $\mu$ anywhere in this optimization — unlike
`MaxSharpe`, it cannot become an error-maximizer over a noisy expected-return
estimate. $\beta^*=0.3$ is fixed *a priori* (a deliberate, modest equity-risk
budget, not fit to the data); $\beta^*=0.5$ below is a sensitivity check, not
a claim that 0.5 is "better." In practice this is **GMV plus just enough
equity exposure to clear the beta floor** — when GMV's own $\beta$ already
clears it, the constraint doesn't bind and `BetaTargetMinVar` *is* GMV (see
`slack_dates` in `src/maplab/models.py`).

In [ ]:
from maplab import BetaTargetMinVar

bmv03 = BetaTargetMinVar(beta_target=0.3)
net_bmv03, wlog_bmv03, diag_bmv03 = ml.backtest(bmv03, simple_returns, panel)

bmv03_group = ml.diagnostics.weights_by_group(wlog_bmv03)
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

fig = h.group_weights_figure(bmv03_group, split_ts)
plt.show()

In [ ]:
ex_ante_beta = pd.Series({
    asof: float(bmv03._beta_vector(panel, asof).reindex(ml.UNIVERSE).to_numpy() @ wlog_bmv03.loc[asof].to_numpy())
    for asof in wlog_bmv03.index
})

fig = h.ex_ante_beta_figure(ex_ante_beta, split_ts)
plt.show()

print(f"n_rebalances={diag_bmv03['n_rebalances']}   "
      f"slack_dates={len(bmv03.slack_dates)}   "
      f"infeasible_dates={len(bmv03.infeasible_dates)}   "
      f"retry_dates={len(bmv03.retry_dates)}")

## Backtest comparison

Fresh instances: `GMV`, `MaxSharpe`, `EqualWeight`, `BetaMinVar(0.3)`,
`BetaMinVar(0.5)`, run through `ml.backtest` exactly as in notebook 01 (10bps
one-way cost, monthly drift-and-rebalance, common warm-up start). Plus two
**$\beta$-matched benchmarks**, $\beta^*\cdot\text{SPY} + (1-\beta^*)\cdot
\text{BIL}$, computed *outside* `ml.backtest` (the backtest's leak guard
forbids BIL as a return column, by design — see `src/maplab/backtest.py`).
These benchmarks carry **two simplifications** the strategies above don't:
(1) **no transaction costs at all** (the strategies pay 10bps one-way on
turnover), and (2) a **continuously (daily) rebalanced** fixed mix, not the
harness's monthly drift-and-rebalance — so the benchmark never drifts off
its target weight between "rebalances." They are a frictionless ceiling, not
an apples-to-apples competitor.

In [ ]:
from maplab import GMV, MaxSharpe, EqualWeight, backtest, summary

strategies = {
    "GMV": GMV(),
    "MaxSharpe": MaxSharpe(),
    "EqualWeight": EqualWeight(),
    "BetaMinVar(0.3)": BetaTargetMinVar(beta_target=0.3),
    "BetaMinVar(0.5)": BetaTargetMinVar(beta_target=0.5),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

scoring_start = results["GMV"]["diag"]["scoring_start"]

for beta_target in (0.3, 0.5):
    bench_ret = (beta_target * simple_returns["SPY"] + (1 - beta_target) * rf_daily).loc[scoring_start:]
    results[f"β-match({beta_target})"] = {"net": bench_ret, "wlog": None, "diag": None, "strat": None}

rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"] if diag is not None else None
    win_defs = {
        "full": (net.index >= net.index.min(), None if turnover is None else turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, None if turnover is None else turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, None if turnover is None else turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel] if (turnover is not None and t_sel is not None) else None
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

bmv = results["BetaMinVar(0.3)"]["strat"]
print(f"BetaMinVar(0.3) slack/infeasible/retry: "
      f"{len(bmv.slack_dates)}/{len(bmv.infeasible_dates)}/{len(bmv.retry_dates)}")
summary_table.round(4)

In [ ]:
fig = h.wealth_figure(results, split_ts)
plt.show()

In [ ]:
fig, dd_dates = h.drawdown_figure(results, split_ts)
plt.show()

print("Full-window annualized turnover (NaN = no turnover concept, the β-match benchmarks):")
display(summary_table.xs("full", level="window")[["ann_turnover"]].round(3))

print("\nMax drawdown: peak -> trough date")
dd_table = pd.DataFrame(dd_dates).set_index("strategy")
dd_table["max_dd"] = (dd_table["max_dd"] * 100).round(2)
dd_table

In [ ]:
print("Sharpe standard error ≈ sqrt((1+SR²/2)/T), T in years (matches notebook 01's convention):")
for name, r in results.items():
    net_full = r["net"]
    net_test = net_full.loc[net_full.index > split_ts]
    sr_full = float(summary_table.loc[(name, "full"), "sharpe"])
    sr_test = float(summary_table.loc[(name, "test (> split)"), "sharpe"])
    T_full = len(net_full) / ml.TRADING_DAYS
    T_test = len(net_test) / ml.TRADING_DAYS
    se_full = ml.inference.sharpe_se(sr_full, T_full)
    se_test = ml.inference.sharpe_se(sr_test, T_test)
    print(f"  {name:18s} full: SR={sr_full:.2f} SE={se_full:.2f}   |   test: SR={sr_test:.2f} SE={se_test:.2f}")

## Ex-post CAPM regression of each strategy

Each strategy's realized daily **simple** excess return (`net - rf_daily`)
regressed on the market's simple excess return (`SPY - BIL`, **simple**
space here — distinct from the log-space MKT used in §2-4 for the exact
factor-portfolio algebra): annualized $\alpha$, $t(\alpha)$, realized $\beta$,
$R^2$.

**Caveat:** statsmodels is not installed in this environment, so this is
plain OLS via `ols_with_intercept` (no Newey-West/HAC correction). Monthly
rebalancing plus daily drift induces autocorrelation in these daily strategy
returns, so the $t$-stats below likely *overstate* significance versus a
HAC-corrected estimate — read them as indicative, not exact.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_strategy_rows = []
for name, r in results.items():
    net = r["net"]
    idx = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    strat_excess = net.loc[idx] - rf_daily.loc[idx]
    mkt_aligned = MKT_simple.loc[idx]
    fit = ml.inference.ols(strat_excess, mkt_aligned.to_frame("MKT"))
    capm_strategy_rows.append({
        "strategy": name,
        "alpha_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha": float(fit["tstat"]["alpha"]),
        "beta": float(fit["coef"]["MKT"]),
        "r2": fit["r2"],
        "n": fit["n"],
    })

capm_strategy_table = pd.DataFrame(capm_strategy_rows).set_index("strategy")
# beta-match rows are a deterministic linear function of MKT (R^2=1) -- their
# t_alpha is numerically meaningless (near-zero coef / near-zero SE).
capm_strategy_table.loc[capm_strategy_table.index.str.startswith("β-match"), "t_alpha"] = np.nan
capm_strategy_table.round(4)

CAPM predicts $\alpha \approx 0$ for `BetaMinVar` — it holds no view on
$\mu$, so any realized alpha here is compensation for the *non-market* risk
GMV minimizes around the beta floor, not a forecast being validated. The
comparison that matters is `BetaMinVar` against its own $\beta$-matched
benchmark (same market exposure, frictionless): a persistently positive
alpha there — if the table above shows one — would say the minimum-variance
tilt around the $\beta$ floor is adding value beyond the $\beta$ exposure
itself; a persistently negative one would say the costs/imperfect-drift of
implementing it aren't worth the diversification. (The $\beta$-match rows'
`t_alpha` is NaN — $R^2=1$ by construction there, so the $t$-stat is
numerically meaningless.)

**TAKEAWAY**
- **CAPM $\mu$ fails algebraically, not just statistically.** The unconstrained CAPM tangency is exactly 100% SPY at all three snapshot dates (relative identity error ~1e-15) — not an estimation artifact but the direct consequence of $\Sigma_x^{-1}(\mu-r_f) = W(W'\Sigma_x W)^{-1}\lambda$ when $W$ is the one-hot SPY vector. Adding TERM/CREDIT only opens the tangency's support to TLT/IEF/HYG (with IEF exactly offsetting HYG, e.g. IEF=+86.5%/HYG=−86.5% at 2022-12-30) — any factor model whose factors are portfolios of the universe can never route information to assets outside those factors' span.
- **`BetaMinVar(0.3)` is well-behaved but adds nothing measurable over simply holding its $\beta$-matched mix.** No corner-hopping, 0/208 slack, infeasible, or retry episodes, and its ex-post realized $\beta$ (0.305) lands almost exactly on its 0.3 target. But against $0.3\cdot\text{SPY}+0.7\cdot\text{BIL}$: Sharpe 0.86 vs 0.83 (SE $\approx$ 0.28, so not distinguishable), a **deeper** max drawdown (−13.9% vs −10.9%), and 59%/yr turnover the frictionless benchmark doesn't pay. Ex-post alpha is small and insignificant (0.51%/yr, $t$=0.96, no HAC) — exactly what CAPM predicts for a strategy that holds no view on $\mu$.
- **None of these Sharpe gaps clear the noise floor.** SE(Sharpe) $\approx$ 0.28 full-window, $\approx$ 0.7 in the 3.3-year test window (same $\sqrt{(1+SR^2/2)/T}$, $T$ in years, as notebook 01) — bigger than every pairwise gap in this notebook. Same caveat as notebook 01, now attached to a $\mu$-free method too: a better risk profile, not a provably better Sharpe.
- **GMV and MaxSharpe show *positive* ex-post alpha at *low* beta** (GMV: $\alpha$=1.2%/yr, $t$=1.66, $\beta$=0.05; MaxSharpe: $\alpha$=3.0%/yr, $t$=2.08, $\beta$=0.10) — consistent with a flatter-than-CAPM security market line (the classic low-beta anomaly, also visible directly in §2's SML plot). Plain OLS, no HAC, one historical path — indicative, not proof.
- **SIZE is mostly noise outside equities (9/13 insignificant), but VALUE is not** (11/13 significant, including rates and credit/commodities) — it carries a duration component, not a clean "value" story outside equities. Pinning a portfolio to either with exact equality constraints (design D3) still corner-hops (VNQ/HYG/DBC/UUP/EEM take turns dominating across the three snapshots), because 252-day loadings are unstable even when non-zero — the same lesson as notebook 01's sample-mean $\mu$: an optimizer leans hard on whatever input is noisiest, significant or not.
- **Next:** CAPM $\mu$ becomes useful as a *prior*, not a rule — combined with views in notebook 03 (Black-Litterman). How much to trust that prior depends on how flat the empirical SML is (§2: empirical slope is only ~24% of the CAPM-implied slope, with a positive intercept CAPM says shouldn't exist) — a prior this far from the theoretical line needs strong views to correct it, or a wide prior variance to admit the uncertainty. One historical path here too; no bootstrap confidence intervals yet (Phase 2).